# 05: the Collatz map through GenerationalLineage, and the control that went wrong

Companion to *A Bit Shift Wearing A Disguise*, Sections 4, 5.7 and 5.9. **[THEORETICAL:CALCULATED]** (the decomposition) and **[OURS:CALCULATED]** (the control). Needs the GenerationalLineage repository next to this one (or `GL_PATH`).

GenerationalLineage ([repository](https://github.com/michaelrendier/GenerationalLineage)) decomposes operations into three primitives: **ADD** (identity 0), **SCALE** (identity 1) and **SIGN** (a ±1 reflection, one bit). The Collatz map was the first real test of the engine (August 2026). The engine and this notebook share an author and an assistant, so this is not an independent check.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / 'engine'))   # run this notebook from CollatzShift/
import e_collatz_gl_decomposition as gld
gl = gld.load_gl()
print('loaded:', gl is not None)
from gl_engine.add_scale_sign import ASS, ASSWord, CAMSHAFT, compose

loaded: True


## Each branch as an engine element

The even branch is SCALE(1/2). The odd branch is ADD(1/2) after SCALE(3/2). The engine's SIGN (sign = −1) appears in neither.

In [2]:
EVEN, ODD = ASS(add=0.0, scale=0.5, sign=1), ASS(add=0.5, scale=1.5, sign=1)
print('even:', EVEN); print('odd :', ODD)
print('odd == ADD(1/2) @ SCALE(3/2):', (ASS.ADD(0.5) @ ASS.SCALE(1.5)) == ODD)
print('SIGN(-1) in either branch:', EVEN.sign == -1 or ODD.sign == -1)
print('engine firing order, innermost first:', CAMSHAFT)

even: x ↦ +1·0.5·x + 0
odd : x ↦ +1·1.5·x + 0.5
odd == ADD(1/2) @ SCALE(3/2): True
SIGN(-1) in either branch: False
engine firing order, innermost first: ('SIGN', 'SCALE', 'ADD')


**Correction (recorded).** The earlier statement in this repository, `T = ADD ∘ SCALE ∘ SIGN`, took SIGN to be the parity bit that *selects* a branch. The engine's SIGN is a reflection and is absent from both branches, so the statement conflated a selector with that primitive. What survives: T is piecewise, its pieces are generated by SCALE and ADD, and one bit selects between them.

## Composing along the true parity word reproduces T^k

In [3]:
from fractions import Fraction
from e_collatz_shift import T, parity_word
bad = n_classes = 0
for k in range(1, 11):
    for j in range(1 << k):
        w = parity_word(j, k); el = ASS.SCALE(1.0)
        for b in w: el = (ODD if b else EVEN) @ el
        n = j + 3 * (1 << k); m = n
        for _ in range(k): m = T(m)
        n_classes += 1; bad += (el.scale != 3 ** sum(w) / 2 ** k) or abs(el(float(n)) - m) > 1e-6 * max(1, m)
print('classes (k<=10):', n_classes, ' mismatches:', bad)

classes (k<=10): 2046  mismatches: 0


## The Long-Path memory (set membership, the rule that decides Recamán's flip)

In [4]:
for n in (27, 97, 871):
    steps, m = [], n
    while m != 1:
        steps.append(ODD if m & 1 else EVEN); m = T(m)
    w = ASSWord(steps=tuple(steps) + (ODD, EVEN), order='collatz', source=compose(*steps))
    col = w.collisions(float(n), tol=0)
    print(f'n={n}: {len(steps)} steps to reach 1; first repeated value {col[0][2]:g} (visited at step {col[0][0]}, again at step {col[0][1]})')

n=27: 70 steps to reach 1; first repeated value 2 (visited at step 69, again at step 71)
n=97: 75 steps to reach 1; first repeated value 2 (visited at step 74, again at step 76)
n=871: 113 steps to reach 1; first repeated value 2 (visited at step 112, again at step 114)


## The control that went wrong (kept)

Taking the parity stream of an orbit as a binary word, its number of Lyndon factors looked higher for Collatz orbits than for random bits. The cells below chase it: the first bit alone moves the count for random bits, and streams that reach 1 end in a nearly forced tail. With both matched the excess is gone. **METHOD error: the control was not matched on the conditioning.**

In [5]:
import random, statistics as st
rng = random.Random(1)
om = lambda bits: gld.duval_count(''.join(map(str, bits)))
for name, first in (('first bit free', None), ('first bit 1', 1), ('first bit 0', 0)):
    xs = []
    for _ in range(3000):
        b = [rng.randint(0, 1) for _ in range(115)]
        if first is not None: b[0] = first
        xs.append(om(b))
    print(f'{name:15s} mean Lyndon factors of 115 random bits: {st.mean(xs):.2f}')

first bit free  mean Lyndon factors of 115 random bits: 6.06


first bit 1     mean Lyndon factors of 115 random bits: 7.12


first bit 0     mean Lyndon factors of 115 random bits: 5.03


In [6]:
res = gld.run(verbose=True, starts=60, draws=100)

Collatz through GenerationalLineage   [A,B: THEORETICAL:CALCULATED; C,D: OURS:CALCULATED]
  A. branch elements: even = x ↦ +1·0.5·x + 0, odd = x ↦ +1·1.5·x + 0.5; engine SIGN (+-1) in either branch: False; odd = ADD(1/2) after SCALE(3/2): True
     composing the branch elements along the true parity word: 8190 classes (k <= 12), mismatches 0; engine firing order (innermost first): ('SIGN', 'SCALE', 'ADD')
  B. Long-Path memory (n: steps to 1, first visit step, revisit step, repeated value): {27: (70, 69, 71, 2.0), 97: (75, 74, 76, 2.0), 871: (113, 112, 114, 2.0), 6171: (165, 164, 166, 2.0), 77031: (221, 220, 222, 2.0)}
  C. Duval vs engine Lyndon omega on 100 random strings, disagreements: 0; rotation-minimal vs primitive words (k,d): {(4, 2): (2, 1), (6, 3): (4, 3), (8, 4): (10, 8)}
  D. Lyndon factors of 115 random bits: first bit free 6.03, first bit 1 7.02, first bit 0 4.99
     mean z of the Collatz streams to 1 (60 odd starts): unmatched +1.51, first bit matched +1.18, first bit 